In [1]:
import os
import getpass
import shutil
import subprocess
from pathlib import Path

# Prompt once per session if HF token is missing (keeps token out of notebook text)
# if not os.environ.get("HF_TOKEN", "").strip():
#     os.environ["HF_TOKEN"] = getpass.getpass("Enter Hugging Face token (HF_TOKEN): ").strip()

# ⚠️ MUST be set BEFORE torch is imported
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pick a writable runtime cache path (JupyterHub-safe)
repo_candidates = ["/home/jovyan/llmpt", os.getcwd()]
repo_root = next((p for p in repo_candidates if os.path.isdir(p)), os.getcwd())

cache_candidates = [
    os.path.join(repo_root, ".runtime_cache"),
    "/tmp/llmpt_runtime_cache",
]

runtime_cache = None
for cand in cache_candidates:
    try:
        Path(cand).mkdir(parents=True, exist_ok=True)
        test_file = Path(cand) / ".write_test"
        test_file.write_text("ok", encoding="utf-8")
        test_file.unlink(missing_ok=True)
        runtime_cache = cand
        break
    except Exception:
        continue

if runtime_cache is None:
    raise RuntimeError("No writable runtime cache path found.")

os.environ["DORADO_RUNTIME_CACHE"] = runtime_cache
os.environ["HF_HOME"] = f"{runtime_cache}/huggingface"
os.environ["HF_HUB_CACHE"] = f"{runtime_cache}/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = f"{runtime_cache}/huggingface/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{runtime_cache}/huggingface/transformers"
os.environ["PIP_CACHE_DIR"] = f"{runtime_cache}/pip"

# Optional: clear old home cache once to reclaim quota
for stale in ["/home/jovyan/.cache/huggingface", "/home/jovyan/.cache/pip"]:
    if os.path.exists(stale):
        shutil.rmtree(stale, ignore_errors=True)

def pick_free_gpus(min_free_gb: int = 16, max_gpus: int = 3) -> list[str]:
    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=index,memory.free",
            "--format=csv,noheader,nounits",
        ]
        rows = subprocess.check_output(cmd, text=True).strip().splitlines()
        parsed = []
        for row in rows:
            idx_str, free_mb_str = [x.strip() for x in row.split(",")[:2]]
            parsed.append((idx_str, int(free_mb_str)))

        keep = [x for x in parsed if x[1] >= min_free_gb * 1024]
        if not keep:
            keep = sorted(parsed, key=lambda x: x[1], reverse=True)[:max_gpus]
        else:
            keep = sorted(keep, key=lambda x: x[1], reverse=True)[:max_gpus]

        return [idx for idx, _ in keep]
    except Exception:
        return ["0"]

selected = pick_free_gpus(min_free_gb=16, max_gpus=5)
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)

# Encourage sharding when multiple GPUs are visible
if len(selected) > 1:
    os.environ["DORADO_MAX_MEMORY_PER_GPU"] = "10GiB"

print(f"🧩 CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"📁 Repo root: {repo_root}")
print(f"💾 Runtime cache: {runtime_cache}")

if not os.environ.get("HF_TOKEN", "").strip():
    print("⚠️ HF_TOKEN is not set. If Qwen/Qwen3-4B is gated/private, model loading will fail.")

# Preflight: check exact model access early
try:
    from huggingface_hub import HfApi
    info = HfApi().model_info("Qwen/Qwen3-4B", token=os.environ.get("HF_TOKEN"))
    print(f"✅ Model access OK: {info.id}")
except Exception as e:
    print(f"⚠️ Model access check failed for Qwen/Qwen3-4B: {e}")

# Minimal fix for HF/filelock incompatibility seen during model download
%pip install -q -U "filelock>=3.12.0"

print("🔍 Checking environment...")
try:
    import torch, transformers, trl, peft, bitsandbytes, datasets
    print(f"✅ All packages present  (torch {torch.__version__}, transformers {transformers.__version__})")
    if torch.cuda.is_available():
        print(f"🎯 Primary visible GPU: {torch.cuda.get_device_name(0)}")
        print(f"🔢 Visible GPU count: {torch.cuda.device_count()}")
except ImportError as e:
    print(f"⚠️ Missing: {e}\n📦 Installing…")
    %pip install -q torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q -U accelerate datasets pyarrow scipy bitsandbytes peft huggingface_hub "transformers>=4.46.0" "trl>=0.11.0" "filelock>=3.12.0"
    print("✅ Done — restart kernel, then re-run this cell.")

🧩 CUDA_VISIBLE_DEVICES=0
📁 Repo root: /home/jovyan/llmpt
💾 Runtime cache: /home/jovyan/llmpt/.runtime_cache
⚠️ HF_TOKEN is not set. If Qwen/Qwen3-4B is gated/private, model loading will fail.


✅ Model access OK: Qwen/Qwen3-4B


Note: you may need to restart the kernel to use updated packages.
🔍 Checking environment...


✅ All packages present  (torch 2.5.1+cu121, transformers 5.2.0)
🎯 Primary visible GPU: NVIDIA GeForce RTX 3090
🔢 Visible GPU count: 1


In [2]:
# 🔎 GPU Diagnostic — run this FIRST to see what's using the GPU
import subprocess, os

print("="*60)
print("GPU STATUS (nvidia-smi)")
print("="*60)
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

GPU STATUS (nvidia-smi)
Tue Feb 24 04:14:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:04:00.0 Off |                  N/A |
|  0%   31C    P8             11W /  350W |    4360MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

In [3]:
import sys, os

# Debug: show where the kernel thinks we are
print(f"CWD: {os.getcwd()}")
print(f"dorado/ exists here? {os.path.isdir('dorado')}")

# Try common locations to find the dorado package
candidates = [
    os.path.abspath("."),                       # current dir
    os.path.expanduser("~/llmpt"),              # home/llmpt
    "/workspaces/llmpt",                        # devcontainer
]

found = False
for p in candidates:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path:
            sys.path.insert(0, p)
        os.chdir(p)  # also set CWD so relative paths work later
        print(f"✅ Found dorado at: {p}")
        found = True
        break

if not found:
    print("❌ Could not find dorado/ — add your repo path to 'candidates' above")
    print(f"   sys.path = {sys.path[:5]}")

CWD: /home/jovyan/llmpt
dorado/ exists here? True
✅ Found dorado at: /home/jovyan/llmpt


In [4]:
# 1) Import the dorado package (auto-reloads on code changes)
%load_ext autoreload
%autoreload 2

from dorado import (
    # config / grid
    build_experiment_grid, estimate_time, make_results_paths,
    DATASET_CONFIG, MODEL_CONFIG, ARCHITECTURE_CONFIG,
    TRAINING_CONFIG, GENERATION_CONFIG, EVAL_CONFIG,
    DUAL_PREFERENCE_CONFIG,
    # stages (can call individually for debugging)
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_rm_training, run_dpo_training, run_full_evaluation,
    # orchestrator
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    # utils
    clear_gpu, set_random_seeds, cleanup_storage,
)

print("✅ dorado package loaded")

✅ dorado package loaded


In [5]:
EXPERIMENTS = build_experiment_grid()
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()

print(f"Experiments: {len(EXPERIMENTS)}")
total_min = sum(estimate_time(e)["total_min"] for e in EXPERIMENTS)
print(f"Est. total:  {total_min:.1f} min ({total_min/60:.1f} h)")
print(f"Results:     {RESULTS_FILE}")

Experiments: 2
Est. total:  12.2 min (0.2 h)
Results:     results/2026-02-24/dorado_results.xlsx


### Run Experiments

The cell below runs every experiment in the grid, checkpointing after each one. Results are exported to Excel at the end.

**To resume** after a crash/restart: just re-run — completed experiments are skipped automatically via the checkpoint file.


In [6]:
! ls

README.md  dorado  dorado_toy.ipynb  nohup.out	plans  results


### 1-Minute Smoke Check

Run this first to validate the full pipeline with tiny settings on a lightweight model.

- Runs **one** experiment only
- Uses very small sample sizes and token limits
- Writes to a separate `results/smoke_check_*` path so your main checkpoint is untouched

In [7]:
# from copy import deepcopy
# from pathlib import Path
# import os
# import pandas as pd
# from datetime import datetime
# from contextlib import redirect_stdout, redirect_stderr

# # Start from your first experiment config and shrink everything aggressively
# smoke = deepcopy(EXPERIMENTS[0])
# smoke.update({
#     "experiment_id": 9990,
#     "base_model": "HuggingFaceTB/SmolLM2-135M",
#     "rm_base_model": "HuggingFaceTB/SmolLM2-135M",
#     "sft_samples": 8,
#     "dpo_pairs": 2,
#     "candidates_per_question": 2,
#     "iterative_dpo_rounds": 1,
#     "sft_epochs": 1,
#     "rm_epochs": 1,
#     "dpo_epochs": 1,
#     "sft_batch_size": 1,
#     "rm_batch_size": 1,
#     "dpo_batch_size": 1,
#     "eval_max_samples": 4,
#     "eval_batch_size": 1,
#     "max_new_tokens_gen": 32,
#     "max_new_tokens_eval": 48,
#     "temperature": 0.2,
#     "quantization_bits": 0,
#     "use_rm_scoring": False,
# })

# # Keep smoke-check outputs isolated from your main experiment artifacts/checkpoints
# smoke_results_dir = Path("results") / "smoke_check"
# os.makedirs(smoke_results_dir, exist_ok=True)

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# smoke_log = smoke_results_dir / f"smoke_check_{timestamp}.log"
# smoke_out = smoke_results_dir / "smoke_check_results.xlsx"

# with open(smoke_log, "w", encoding="utf-8") as logf, redirect_stdout(logf), redirect_stderr(logf):
#     print("🚦 Running 1-minute smoke check with config:")
#     for k in [
#         "base_model", "sft_samples", "dpo_pairs", "candidates_per_question",
#         "sft_epochs", "dpo_epochs", "eval_max_samples", "max_new_tokens_gen",
#         "max_new_tokens_eval", "use_rm_scoring"
#     ]:
#         print(f"  {k}: {smoke[k]}")

#     smoke_result = run_single_experiment(smoke)
#     smoke_df = pd.DataFrame([smoke_result])
#     smoke_df.to_excel(smoke_out, index=False)

#     print(f"\n✅ Smoke check status: {smoke_result.get('status')}")
#     print(f"⏱️ Runtime: {smoke_result.get('runtime_minutes', 0):.2f} min")
#     print(f"📄 Saved results: {smoke_out}")

#     print("\n🧹 Cleaning smoke-check artifacts...")
#     cleanup_artifacts()

# print(f"✅ Smoke check finished. Log: {smoke_log}")
# print(f"📄 Results: {smoke_out}")
# smoke_df

In [8]:
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr
import traceback

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
full_log = Path(RESULTS_FILE).with_name(f"dorado_full_run_{timestamp}.log")
full_log.parent.mkdir(parents=True, exist_ok=True)

run_error = None
results_df = None

try:
    with open(full_log, "w", encoding="utf-8") as logf, redirect_stdout(logf), redirect_stderr(logf):
        print("🚀 Starting full Dorado run")
        print(f"Results file: {RESULTS_FILE}")
        print(f"Checkpoint file: {CHECKPOINT_FILE}")
        print(f"Experiments: {len(EXPERIMENTS)}")
        try:
            results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)
        except BaseException as e:
            run_error = e
            traceback.print_exc()
        finally:
            logf.flush()
except Exception as e:
    run_error = e
    print(f"❌ Could not initialize logging at {full_log}: {e}")

print(f"📝 Full-run log: {full_log}")
if run_error is not None:
    print(f"❌ Full run failed. Check log for details: {full_log}")
else:
    print(f"✅ Full run completed. Results: {RESULTS_FILE}")

results_df

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
10,1.974295
20,1.838851
30,1.861499
40,1.904104
50,1.663456
60,1.746794
70,1.697922
80,1.612434
90,1.631341
100,1.525357


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating Candidates:   0%|          | 0/50 [00:00<?, ?it/s]

...

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


,Model,Accuracy %,95% CI Lower,95% CI Upper,CI Width,n
0,BASE,21.0,15.5,27.0,11.5,200
1,SFT,16.5,11.5,21.0,9.5,200
2,DORADO,21.0,15.5,27.0,11.5,200


📝 Full-run log: results/2026-02-24/dorado_full_run_20260224_041449.log
✅ Full run completed. Results: results/2026-02-24/dorado_results.xlsx


,experiment_id,status,runtime_minutes,error,sft_samples,dpo_pairs,candidates_per_question,sft_dataset_name,eval_split,eval_max_samples,...,final_avg_rm_score,round_metrics,base_ci_lower,base_ci_upper,sft_ci_lower,sft_ci_upper,dorado_ci_lower,dorado_ci_upper,warning_count,pipeline_warnings
0,0,success,114.414568,None,500,50,4,tatsu-lab/alpaca,test,200,...,0.000000,"[{'round': 1, 'num_pairs': 55, 'correct_incorr...",0.155,0.27,0.115,0.21,0.155,0.27,3,Generation: 24/200 (12%) candidates lack '####...
1,1,success,117.233880,None,500,50,4,tatsu-lab/alpaca,test,200,...,0.837637,"[{'round': 1, 'num_pairs': 100, 'correct_incor...",0.155,0.27,0.115,0.21,0.155,0.27,4,Generation: 24/200 (12%) candidates lack '####...
